In [1]:
import os
import re
import json
import argparse
import numpy as np
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from scipy.optimize import curve_fit, minimize_scalar
from scipy.stats import bootstrap

In [2]:
# ============================================================
# Paths / data loading
# ============================================================

def parse_tag(filename):
    """Extract (L, beta, sector) from filename tag."""
    m = re.search(r"L(\d+)_beta([0-9.]+)_sector(\d)(\d)",
                  os.path.basename(filename))
    if m is None:
        return None
    L      = int(m.group(1))
    beta   = float(m.group(2))
    sector = (int(m.group(3)), int(m.group(4)))
    return L, beta, sector


def load_obs(datadir):
    """
    Load all .obs.npy files in datadir.

    Returns
    -------
    data : dict  keyed by (L, beta, sector)
              value: dict of 1-D arrays, one per observable
    """
    OBS_KEYS = ["m", "m2", "m4", "abm", "seam_x", "seam_y"]
    data = {}

    for fname in sorted(os.listdir(datadir)):
        if not fname.endswith(".obs.npy"):
            continue

        tag = parse_tag(fname)
        if tag is None:
            continue

        arr = np.load(os.path.join(datadir, fname))   # (steps, n_obs)

        data[tag] = {k: arr[:, i] for i, k in enumerate(OBS_KEYS)}

    return data

In [3]:
import numpy as np



def tau_int(x, c=6.0):
    """
    Madras-Sokal windowed estimate of the integrated autocorrelation time.
    Returns (tau, W, error_estimate).

    tau_int = 0.5 + sum_{t=1}^{W} rho(t)
    Window W chosen self-consistently: W = min{ t : t >= c * tau_int(t) }
    Jackknife error is computed iteratively to avoid recursion.
    """
    ac = autocorr(x)
    
    # Compute tau_int with windowing
    tau = 0.5
    W = len(ac) - 1
    for t in range(1, len(ac)):
        tau += ac[t]
        if t >= c * tau:
            W = t
            break

    # Jackknife error estimate
    n = len(x)
    block = max(1, int(2 * tau))
    nb = n // block

    if nb < 4:
        err = np.nan
    else:
        # Split into blocks and compute tau leaving out each block
        tau_jk = np.zeros(nb)
        for k in range(nb):
            # Concatenate all blocks except the k-th
            x_jk = np.concatenate([x[:k*block], x[(k+1)*block:]])
            ac_jk = autocorr(x_jk)
            
            # Windowed sum for this jackknife sample
            tau_k = 0.5
            for t in range(1, len(ac_jk)):
                tau_k += ac_jk[t]
                if t >= c * tau_k:
                    break
            tau_jk[k] = tau_k

        # Jackknife variance
        err = np.sqrt((nb - 1) * np.var(tau_jk, ddof=0))

    return tau, W, err

In [4]:
# ============================================================
# Autocorrelation & integrated autocorrelation time
# ============================================================

def autocorr(x):
    """Normalised autocorrelation function via FFT."""
    x  = x - x.mean()
    n  = len(x)
    f  = np.fft.rfft(x, n=2 * n)
    ac = np.fft.irfft(f * np.conj(f))[:n]
    ac /= ac[0]
    return ac


def tau_int(x, c=6.0):
    """
    Madras-Sokal windowed estimate of the integrated autocorrelation time.

    tau_int = 0.5 + sum_{t=1}^{W} rho(t)

    Window W chosen self-consistently:  W = min{ t : t >= c * tau_int(t) }.
    Returns (tau, W, error_estimate).
    """
    ac  = autocorr(x)
    tau = 0.5
    for t in range(1, len(ac)):
        tau += ac[t]
        if t >= c * tau:
            W = t
            break
    else:
        W = len(ac) - 1

    # Jackknife error on tau_int
    n     = len(x)
    block = max(1, int(2 * tau))
    nb    = n // block
    if nb < 4:
        err = np.nan
    else:
        blocks = x[: nb * block].reshape(nb, block)
        tau_jk = np.array([
            tau_int(np.concatenate([blocks[:k], blocks[k+1:]], axis=0).ravel(), c=c)[0]
            for k in range(nb)
        ])
        err = np.sqrt((nb - 1) * np.var(tau_jk, ddof=0))

    return tau, W, err

# ============================================================
# Autocorrelation & integrated autocorrelation time
# ============================================================

def autocorr(x):
    """Normalised autocorrelation function via FFT."""
    x = x - x.mean()
    n = len(x)
    f = np.fft.rfft(x, n=2 * n)
    ac = np.fft.irfft(f * np.conj(f))[:n]
    ac /= ac[0]
    return ac

def tau_int(x, c=6.0):
    """
    Madras-Sokal windowed estimate of the integrated autocorrelation time with jack-knife error estimate.

    tau_int = 0.5 + sum_{t=1}^{W} rho(t)

    Window W chosen self-consistently:  W = min{ t : t >= c * tau_int(t) }.
    Returns (tau, W, error_estimate).
    """
    # FFT autocorrelation
    def autocorr(x):
        x = x - x.mean()
        n = len(x)
        f = np.fft.rfft(x, n=2*n)
        ac = np.fft.irfft(f * np.conj(f))[:n]
        ac /= ac[0]
        return ac

    # Compute tau_int for full data
    ac = autocorr(x)
    tau = 0.5
    W = len(ac) - 1
    for t in range(1, len(ac)):
        tau += ac[t]
        if t >= c * tau:
            W = t
            break

    # Determine block size and number of blocks
    n = len(x)
    block = max(1, int(2 * tau))
    nb = n // block

    if nb < 4:
        err = np.nan
    else:
        # Reshape into blocks
        blocks = x[:nb*block].reshape(nb, block)
        
        # Precompute cumulative sums for each block to avoid repeated concatenation
        # Each row in cum_blocks is sum of all elements except that block
        tau_jk = np.zeros(nb)
        for k in range(nb):
            x_jk = np.delete(blocks, k, axis=0).ravel()  # remove block k
            ac_jk = autocorr(x_jk)
            
            # Windowed tau
            tau_k = 0.5
            for t in range(1, len(ac_jk)):
                tau_k += ac_jk[t]
                if t >= c * tau_k:
                    break
            tau_jk[k] = tau_k

        err = np.sqrt((nb-1) * np.var(tau_jk, ddof=0))

    return tau, W, err

# ============================================================
# Statistical estimators
# ============================================================

def jackknife(f, *arrays, block=1):
    """
    Jackknife mean and error for function  f(*arrays).
    Uses blocking to account for correlations.
    """
    n  = len(arrays[0])
    nb = n // block
    blocked = [a[: nb * block].reshape(nb, block).mean(axis=1) for a in arrays]

    full_val = f(*[a.mean() for a in blocked])

    jk_vals = np.array([
        f(*[np.concatenate([b[:k], b[k+1:]]).mean() for b in blocked])
        for k in range(nb)
    ])

    mean = (nb * full_val) - (nb - 1) * jk_vals.mean()
    err  = np.sqrt((nb - 1) * np.var(jk_vals, ddof=0))
    return mean, err


def binder(m2, m4):
    """Binder cumulant U4 = 1 - <m^4> / (3 <m^2>^2)."""
    return 1.0 - m4.mean() / (3.0 * m2.mean() ** 2)


def susceptibility(m2, abm, beta, L):
    """chi = beta * L^2 * (<m^2> - <|m|>^2)"""
    return beta * L**2 * (m2.mean() - abm.mean()**2)

In [5]:






# ============================================================
# FSS scaling functions
# ============================================================

def binder_scaling(t, a, b, c):
    """
    U4(t) = a + b*t + c*t^2    (polynomial in reduced temp near crossing)
    Used to interpolate Binder curves and find crossing.
    """
    return a + b * t + c * t ** 2


def obs_fss(L, t, x0, nu, A, B):
    """
    Generic FSS ansatz for an observable O with scaling dimension x0:

        O(L, t) = L^(x0/nu) * f(t * L^(1/nu))
                ~ L^(x0/nu) * (A + B * t * L^(1/nu))

    x0 : scaling dimension of O   (e.g. 2-eta for chi,  1/nu for dU/dt)
    """
    u = t * L ** (1.0 / nu)
    return L ** (x0 / nu) * (A + B * u)





# ============================================================
# Step 2: nu from collapse of dU/d(beta)
# ============================================================

def fit_nu(data, sizes, beta_c, sector=(0, 0)):
    """
    At criticality,  d U4 / d(beta)  ~  L^{1/nu}.
    Estimate the derivative numerically from finite differences.
    """
    print("\n--- Fit nu ---")

    L_arr, deriv_arr, err_arr = [], [], []

    # Group by L
    from collections import defaultdict
    grouped = defaultdict(dict)

    for (L, beta, sec), obs in data.items():
        if sec != sector:
            continue
        block = max(1, int(2 * tau_int(obs["m2"])[0]))
        u, e  = jackknife(binder, obs["m2"], obs["m4"], block=block)
        grouped[L][beta] = (u, e)

    for L, bd in grouped.items():
        betas = np.array(sorted(bd.keys()))
        u4s   = np.array([bd[b][0] for b in betas])

        # Numerical derivative at beta_c via central differences
        dU = np.gradient(u4s, betas)
        # Interpolate to beta_c
        dU_c = np.interp(beta_c, betas, dU)

        # Error: propagate jackknife errors numerically
        errs  = np.array([bd[b][1] for b in betas])
        dU_err = np.sqrt(np.sum((np.gradient(errs, betas)) ** 2))

        L_arr.append(L)
        deriv_arr.append(dU_c)
        err_arr.append(dU_err + 1e-8)

    L_arr    = np.array(L_arr, dtype=float)
    deriv_arr = np.array(deriv_arr)
    err_arr   = np.array(err_arr)

    # Fit  log(dU/dbeta) = (1/nu) log(L) + const
    log_L = np.log(L_arr)
    log_d = np.log(np.abs(deriv_arr))

    coeffs = np.polyfit(log_L, log_d, 1)
    nu_fit = coeffs[0]

    print(f"  1/nu = {nu_fit:.4f}  =>  nu = {1.0/nu_fit:.4f}  (exact: 1.0)")

    return 1.0 / nu_fit, L_arr, deriv_arr, err_arr


# ============================================================
# Step 3: eta from susceptibility scaling  chi ~ L^{2-eta}
# ============================================================

def fit_eta(data, sizes, beta_c, sector=(0, 0)):
    """
    chi(beta_c, L)  ~  L^{2-eta}

    We evaluate chi at the nearest beta to beta_c for each L.
    """
    print("\n--- Fit eta ---")

    L_arr, chi_arr, err_arr = [], [], []

    for L in sizes:
        best_db = np.inf
        best_chi = None
        best_err = None

        for (Lk, betak, seck), obs in data.items():
            if Lk != L or seck != sector:
                continue
            db = abs(betak - beta_c)
            if db < best_db:
                best_db = db
                block   = max(1, int(2 * tau_int(obs["m2"])[0]))
                chi     = susceptibility(obs["m2"], obs["abm"], beta_c, L)
                # jackknife error for chi
                def chi_fn(m2, abm):
                    return beta_c * L**2 * (m2 - abm**2)
                _, e = jackknife(chi_fn, obs["m2"], obs["abm"], block=block)
                best_chi = chi
                best_err = e

        if best_chi is not None:
            L_arr.append(L)
            chi_arr.append(best_chi)
            err_arr.append(best_err + 1e-8)

    L_arr   = np.array(L_arr, dtype=float)
    chi_arr = np.array(chi_arr)
    err_arr = np.array(err_arr)

    log_L   = np.log(L_arr)
    log_chi = np.log(chi_arr)

    coeffs = np.polyfit(log_L, log_chi, 1, w=1.0 / err_arr)
    two_minus_eta = coeffs[0]
    eta_fit       = 2.0 - two_minus_eta

    print(f"  2-eta = {two_minus_eta:.4f}  =>  eta = {eta_fit:.4f}  (exact: 0.25)")

    return eta_fit, L_arr, chi_arr, err_arr


# ============================================================
# Step 4: Conformal dimensions from operator scaling
# ============================================================

def conformal_dimensions(eta, nu):
    """
    Standard Ising conformal dimensions from critical exponents.

    Scaling dimensions (= conformal weights h + h-bar for spinless ops):

        x_sigma = (2 - eta) / 2  =  1/8   (spin field)
        x_eps   = d - 1/nu       =  1     (energy density, d=2)

    Conformal weights:
        h = x/2   (for left = right)
    """
    x_sigma = (2.0 - eta) / 2.0
    x_eps   = 2.0 - 1.0 / nu

    print("\n--- Conformal dimensions (bulk) ---")
    print(f"  x_sigma = {x_sigma:.4f}  (exact: 0.125)")
    print(f"  x_eps   = {x_eps:.4f}  (exact: 1.000)")
    print(f"  h_sigma = {x_sigma/2:.4f}  (exact: 0.0625)")
    print(f"  h_eps   = {x_eps/2:.4f}  (exact: 0.500)")

    return dict(
        x_sigma=x_sigma,
        x_eps=x_eps,
        h_sigma=x_sigma / 2.0,
        h_eps=x_eps / 2.0,
    )


# ============================================================
# Step 5: Seam (twist) conformal dimension from free energy ratio
# ============================================================

def seam_conformal_dimension(data, sizes, beta_c):
    """
    The insertion of a defect seam (twist operator) costs a free energy:

        Delta F = F_{twisted} - F_{untwisted}
                = -log( Z_{tx=1} / Z_{tx=0} )

    On a torus of size L x L, CFT predicts:

        Z_{sector} = sum_{primary phi} n_{phi} * chi_{h}(q) * chi_{hbar}(qbar)

    For an L x L torus with modular parameter tau = i (square):

        -log( Z_{10} / Z_{00} )  ~  2*pi * x_seam / L   as L -> inf

    More precisely, in the leading term:

        ln( Z_{10} / Z_{00} )  =  -2*pi * x_seam / L  +  O(1/L^2)

    We estimate  Z_{sector} / Z_{00}  via the ratio of partition functions,
    accessible through thermodynamic integration or — more practically —
    from the free-energy difference via:

        <dS/d(twist)>  at half-twist

    A cleaner route available from our data uses the ratio of the
    seam bond energy  E_seam  between sectors via:

        ln Z_{twist} - ln Z_{0}  =  integral_0^1 d(lambda) <E_seam(lambda)>

    where lambda interpolates J -> -J on the seam bonds.

    Since we only have the two endpoints (lambda=0 and lambda=1),
    we use the midpoint / trapezoidal approximation:

        Delta F  ~  -( <E_seam>_{twist} - <E_seam>_{no-twist} )

    Then fit   Delta F(L)  ~  (2*pi/L) * x_seam.

    This is approximate; for a more precise estimate use
    parallel tempering between sectors.
    """
    print("\n--- Seam conformal dimension ---")

    L_arr, dF_arr = [], []

    for L in sizes:
        # Find beta closest to beta_c for this L
        def get_seam_energy(sector_key):
            best_db = np.inf
            best_val = None
            for (Lk, betak, seck), obs in data.items():
                if Lk != L or seck != sector_key:
                    continue
                db = abs(betak - beta_c)
                if db < best_db:
                    best_db  = db
                    best_val = obs["seam_x"].mean()  # <s_{i,Lx-1} s_{i,0}>
            return best_val

        E00 = get_seam_energy((0, 0))
        E10 = get_seam_energy((1, 0))

        if E00 is None or E10 is None:
            continue

        # Bond energy density across x-seam:
        #   e_seam = J * <s * s'>   (J=+1 for (0,0), J=-1 for (1,0))
        # Free energy cost of inserting the twist ~ integral of <E_seam>
        # Trapezoidal over lambda:  0 -> 1 means J: +1 -> -1
        # <E_seam(lambda=0)> = +J * E00,  <E_seam(lambda=1)> = -J * E10
        dF = -0.5 * L * (E00 - (-E10))   # L factor: sum over Ly bonds

        dF_arr.append(dF)
        L_arr.append(L)
        print(f"  L={L:3d}   Delta_F = {dF:.4f}")

    if len(L_arr) < 2:
        print("  (not enough L values)")
        return None

    L_arr  = np.array(L_arr, dtype=float)
    dF_arr = np.array(dF_arr)

    # Fit  Delta_F = 2*pi * x_seam / L  =>  x_seam = Delta_F * L / (2*pi)
    x_seam_arr = dF_arr * L_arr / (2.0 * np.pi)

    print(f"  x_seam estimates per L: {x_seam_arr}")

    # Extrapolate to L->inf using 1/L^2 corrections:
    #   x_seam(L) = x_seam_inf + c / L^2
    if len(L_arr) >= 3:
        def model(L, x_inf, c):
            return x_inf + c / L**2
        try:
            popt, pcov = curve_fit(model, L_arr, x_seam_arr)
            x_seam_inf = popt[0]
            x_seam_err = np.sqrt(pcov[0, 0])
        except RuntimeError:
            x_seam_inf = x_seam_arr.mean()
            x_seam_err = x_seam_arr.std()
    else:
        x_seam_inf = x_seam_arr.mean()
        x_seam_err = x_seam_arr.std()

    print(f"\n  x_seam = {x_seam_inf:.4f} +/- {x_seam_err:.4f}")
    print(f"  (exact for Z2 twist: x_seam = 1/8 = 0.125  [spin field])")
    print(f"  (the twist operator is the disorder field mu, x_mu = 1/8)")

    return dict(
        L=L_arr,
        dF=dF_arr,
        x_seam_per_L=x_seam_arr,
        x_seam=x_seam_inf,
        x_seam_err=x_seam_err,
    )


# ============================================================
# Step 6: Autocorrelation times per sector
# ============================================================

def compute_tau_int_all(data, beta_c, sizes):
    """
    Compute tau_int for magnetization in each sector at beta ~ beta_c.
    """
    print("\n--- Integrated autocorrelation times at beta_c ---")

    results = {}

    for sector in [(0,0),(1,0),(0,1),(1,1)]:
        for L in sizes:
            best_db = np.inf
            best_obs = None

            for (Lk, betak, seck), obs in data.items():
                if Lk != L or seck != sector:
                    continue
                db = abs(betak - beta_c)
                if db < best_db:
                    best_db  = db
                    best_obs = obs

            if best_obs is None:
                continue

            tau, W, err = tau_int(best_obs["abm"])
            results[(L, sector)] = (tau, W, err)

            print(f"  sector={sector}  L={L:3d}  tau_int={tau:.1f}  W={W}  err={err:.1f}")

    return results


# ============================================================
# Plotting
# ============================================================

def plot_binder(binder_results, beta_c, outdir):
    fig, ax = plt.subplots(figsize=(7, 5))
    cmap = plt.get_cmap("viridis")
    sorted_L = sorted(binder_results.keys())

    for i, L in enumerate(sorted_L):
        betas, U4s, popt = binder_results[L]
        c = cmap(i / len(sorted_L))
        ax.scatter(betas, U4s, color=c, s=20, label=f"L={L}")
        b_fine = np.linspace(betas.min(), betas.max(), 200)
        ax.plot(b_fine, binder_scaling(b_fine, *popt), color=c, lw=1.2)

    ax.axvline(beta_c, color="red", ls="--", lw=1.5, label=f"β_c={beta_c:.5f}")
    ax.set_xlabel("β")
    ax.set_ylabel("U₄")
    ax.set_title("Binder Cumulant")
    ax.legend(fontsize=8)
    fig.tight_layout()
    fig.savefig(os.path.join(outdir, "binder.png"), dpi=150)
    plt.close(fig)


def plot_scaling(L_arr, obs_arr, err_arr, slope, intercept, label, ylabel, outdir, fname):
    fig, ax = plt.subplots(figsize=(6, 4))
    ax.errorbar(np.log(L_arr), np.log(obs_arr),
                yerr=err_arr / (obs_arr + 1e-12),
                fmt="o", color="steelblue", capsize=4)
    x_line = np.linspace(np.log(L_arr.min()), np.log(L_arr.max()), 100)
    ax.plot(x_line, slope * x_line + intercept, "r--", lw=1.5,
            label=f"slope = {slope:.3f}")
    ax.set_xlabel("log L")
    ax.set_ylabel(f"log {ylabel}")
    ax.set_title(label)
    ax.legend()
    fig.tight_layout()
    fig.savefig(os.path.join(outdir, fname), dpi=150)
    plt.close(fig)


def plot_tau_scaling(tau_results, outdir):
    """log tau vs log L for each sector."""
    fig, ax = plt.subplots(figsize=(7, 5))
    sectors = [(0,0),(1,0),(0,1),(1,1)]
    labels  = ["(0,0)","(1,0)","(0,1)","(1,1)"]
    markers = ["o","s","^","D"]

    for sec, lab, mk in zip(sectors, labels, markers):
        Ls, taus = [], []
        for (L, s), (tau, W, err) in tau_results.items():
            if s == sec:
                Ls.append(L)
                taus.append(tau)
        if len(Ls) < 2:
            continue
        Ls   = np.array(Ls, dtype=float)
        taus = np.array(taus)
        ax.plot(np.log(Ls), np.log(taus), marker=mk, label=f"sector {lab}")

    ax.set_xlabel("log L")
    ax.set_ylabel("log τ_int")
    ax.set_title("Critical slowing down by sector")
    ax.legend()
    fig.tight_layout()
    fig.savefig(os.path.join(outdir, "tau_scaling.png"), dpi=150)
    plt.close(fig)

In [6]:
datadir = "ising_critical_scan_long"
outdir = "fss_results"
bulk_sector = (0,0)

In [7]:
data = load_obs(datadir)
os.makedirs(outdir, exist_ok=True)
print(f"  {len(data)} (L, beta, sector) entries loaded.")
sizes = sorted({L for (L, b, s) in data.keys()})
print(f"  Sizes: {sizes}")

  320 (L, beta, sector) entries loaded.
  Sizes: [24, 32, 48, 64]


In [13]:
remove_before = 7000
truncated_data = {}
for key, val in data.items():
    temp = {}
    for obs in val.keys():
        temp[obs] = val[obs][remove_before:]
    truncated_data[key] = temp

In [14]:
# ============================================================
# Step 1: beta_c from Binder cumulant crossings
# ============================================================

def find_beta_c(data, sizes, sector=(0, 0)):
    """
    Fit U4(beta) for each L, find pairwise crossings, return mean.
    """
    print("\n--- Binder crossing ---")

    results = {}

    for L in sizes:
        betas, U4s = [], []

        for (Lk, betak, seck), obs in data.items():
            if Lk != L or seck != sector:
                continue
            block = max(1, int(2 * tau_int(obs["m2"])[0]))
            u, _  = jackknife(binder, obs["m2"], obs["m4"], block=block)
            betas.append(betak)
            U4s.append(u)

        if len(betas) < 3:
            continue

        idx    = np.argsort(betas)
        betas  = np.array(betas)[idx]
        U4s    = np.array(U4s)[idx]

        try:
            popt, _ = curve_fit(binder_scaling, betas, U4s, p0=[0.6, 0.0, 0.0])
            results[L] = (betas, U4s, popt)
        except RuntimeError:
            pass

    # Pairwise crossings between consecutive L pairs
    crossings = []
    sorted_L  = sorted(results.keys())

    for i in range(len(sorted_L) - 1):
        L1, L2 = sorted_L[i], sorted_L[i + 1]
        b1, _, p1 = results[L1]
        b2, _, p2 = results[L2]

        beta_range = (max(b1.min(), b2.min()), min(b1.max(), b2.max()))

        def diff(beta):
            return abs(binder_scaling(beta, *p1) - binder_scaling(beta, *p2))

        res = minimize_scalar(diff, bounds=beta_range, method="bounded")
        if res.success:
            crossings.append(res.x)
            print(f"  L={L1:3d} x L={L2:3d}  beta_c = {res.x:.6f}")

    beta_c = np.mean(crossings) if crossings else 0.44069
    print(f"  => beta_c = {beta_c:.6f}  (mean of {len(crossings)} crossings)")
    return beta_c, results

In [15]:
beta_c, binder_res = find_beta_c(truncated_data, sizes, sector=bulk_sector)
nu,   L_nu,  dU_nu,  err_nu  = fit_nu(truncated_data,  sizes, beta_c, sector=bulk_sector)
eta,  L_chi, chi,    err_chi = fit_eta(truncated_data,  sizes, beta_c, sector=bulk_sector)
dims = conformal_dimensions(eta, nu)

seam = seam_conformal_dimension(truncated_data, sizes, beta_c)

tau_res = compute_tau_int_all(truncated_data, beta_c, sizes)


--- Binder crossing ---
  L= 24 x L= 32  beta_c = 0.440404
  L= 32 x L= 48  beta_c = 0.441194
  L= 48 x L= 64  beta_c = 0.439971
  => beta_c = 0.440523  (mean of 3 crossings)

--- Fit nu ---
  1/nu = 1.0004  =>  nu = 0.9996  (exact: 1.0)

--- Fit eta ---
  2-eta = 1.8008  =>  eta = 0.1992  (exact: 0.25)

--- Conformal dimensions (bulk) ---
  x_sigma = 0.9004  (exact: 0.125)
  x_eps   = 0.9996  (exact: 1.000)
  h_sigma = 0.4502  (exact: 0.0625)
  h_eps   = 0.4998  (exact: 0.500)

--- Seam conformal dimension ---
  L= 24   Delta_F = -0.5868
  L= 32   Delta_F = -0.6112
  L= 48   Delta_F = -0.5423
  L= 64   Delta_F = -0.4772
  x_seam estimates per L: [-2.24129106 -3.11257974 -4.14291954 -4.86024847]

  x_seam = -5.0145 +/- 0.2619
  (exact for Z2 twist: x_seam = 1/8 = 0.125  [spin field])
  (the twist operator is the disorder field mu, x_mu = 1/8)

--- Integrated autocorrelation times at beta_c ---
  sector=(0, 0)  L= 24  tau_int=2.3  W=14  err=0.2
  sector=(0, 0)  L= 32  tau_int=4.2  W=26

In [16]:
# ----------------------------------------------------------
# Plots
# ----------------------------------------------------------
plot_binder(binder_res, 0.4407, outdir)

log_L_nu  = np.log(L_nu)
log_dU    = np.log(np.abs(dU_nu))
c_nu      = np.polyfit(log_L_nu, log_dU, 1)
plot_scaling(L_nu, np.abs(dU_nu), err_nu,
             c_nu[0], c_nu[1],
             "dU₄/dβ vs L  (slope = 1/ν)", "dU₄/dβ",
             outdir, "nu_scaling.png")

log_L_chi = np.log(L_chi)
log_chi   = np.log(chi)
c_chi     = np.polyfit(log_L_chi, log_chi, 1)
plot_scaling(L_chi, chi, err_chi,
             c_chi[0], c_chi[1],
             "χ vs L  (slope = 2−η)", "χ",
             outdir, "eta_scaling.png")

plot_tau_scaling(tau_res, outdir)

In [17]:
summary = dict(
        beta_c      = float(beta_c),
        nu          = float(nu),
        eta         = float(eta),
        **{k: float(v) for k, v in dims.items()},
        x_seam      = float(seam["x_seam"])      if seam else None,
        x_seam_err  = float(seam["x_seam_err"])  if seam else None,
    )

import json
with open(os.path.join(outdir, "summary.json"), "w") as f:
    json.dump(summary, f, indent=2)

print("\n========== SUMMARY ==========")
for k, v in summary.items():
    print(f"  {k:20s} = {v}")

print(f"\nPlots and summary written to: {outdir}/")


========== SUMMARY ==========
  beta_c               = 0.44052299913633125
  nu                   = 0.999594833150944
  eta                  = 0.19915844264331195
  x_sigma              = 0.900420778678344
  x_eps                = 0.9995946689242292
  h_sigma              = 0.450210389339172
  h_eps                = 0.4997973344621146
  x_seam               = -5.014527426462873
  x_seam_err           = 0.26192345309327014

Plots and summary written to: fss_results/


In [ ]:
def seam_conformal_dimension(data, sizes, beta_c):
    """
    Extract x_seam from the power-law decay of the seam bond correlator
    in sector (1,0) at criticality:

        <|seam_x|>(L)  ~  L^{-2 * x_seam}

    For the Z2 twist (disorder field mu):  x_seam = 1/8, so slope = -1/4.

    Also extract from sector (0,1) via seam_y as a cross-check.
    """
    print("\n--- Seam conformal dimension ---")

    results = {}

    for seam_sector, obs_key in [((1, 0), "seam_x"), ((0, 1), "seam_y")]:

        L_arr, s_arr, e_arr = [], [], []

        for L in sizes:
            best_db  = np.inf
            best_val = None
            best_err = None

            for (Lk, betak, seck), obs in data.items():
                if Lk != L or seck != seam_sector:
                    continue
                db = abs(betak - beta_c)
                if db < best_db:
                    best_db  = db
                    # Use equilibrated portion only
                    t_burn, _ = find_equilibration(obs[obs_key], method="all")
                    x_eq      = np.abs(obs[obs_key][t_burn:])
                    if len(x_eq) < 20:
                        continue
                    block     = max(1, int(2 * tau_int(x_eq)[0]))
                    nb        = len(x_eq) // block
                    blocks    = x_eq[: nb * block].reshape(nb, block).mean(axis=1)
                    best_val  = blocks.mean()
                    best_err  = blocks.std(ddof=1) / np.sqrt(nb)

            if best_val is not None:
                L_arr.append(L)
                s_arr.append(best_val)
                e_arr.append(best_err + 1e-10)
                print(f"  sector={seam_sector}  L={L:3d}  "
                      f"<|{obs_key}|> = {best_val:.6f} +/- {best_err:.6f}")

        if len(L_arr) < 2:
            continue

        L_arr = np.array(L_arr, dtype=float)
        s_arr = np.array(s_arr)
        e_arr = np.array(e_arr)

        # Simple power law first:  log s = -2*x * log L + const
        log_L = np.log(L_arr)
        log_s = np.log(s_arr)
        w     = s_arr / e_arr   # weights

        coeffs      = np.polyfit(log_L, log_s, 1, w=w)
        slope_naive = coeffs[0]
        x_naive     = -slope_naive / 2.0

        # With correction to scaling:  s(L) = A * L^{-2x} * (1 + B * L^{-2})
        def model_corrected(L, two_x, A, B):
            return A * L**(-two_x) * (1.0 + B * L**(-2))

        try:
            popt, pcov = curve_fit(
                model_corrected, L_arr, s_arr,
                p0=[0.25, s_arr[-1] * L_arr[-1]**0.25, 0.1],
                sigma=e_arr, absolute_sigma=True,
                maxfev=10000
            )
            x_corrected = popt[0] / 2.0
            x_err       = np.sqrt(pcov[0, 0]) / 2.0
        except RuntimeError:
            x_corrected = x_naive
            x_err       = np.nan

        print(f"\n  sector={seam_sector}  naive slope={slope_naive:.4f}"
              f"  x_seam (naive)     = {x_naive:.4f}")
        print(f"  sector={seam_sector}  "
              f"  x_seam (corrected) = {x_corrected:.4f} +/- {x_err:.4f}"
              f"  (exact: 0.1250)")

        results[seam_sector] = dict(
            L=L_arr, s=s_arr, e=e_arr,
            x_naive=x_naive,
            x_corrected=x_corrected,
            x_err=x_err,
        )

    return results